# Multilingual RAG for Indian Languages — A Practical Blueprint

**Companion notebook to the article:** *From English-First to India-First: A Stage-by-Stage RAG Rebuild, With Code*

This notebook walks through building a Retrieval-Augmented Generation (RAG) pipeline that is
deliberately designed for Indian languages (Hindi, Bhojpuri, Maithili, and others), rather than
assuming an English-tuned pipeline generalizes.

Every stage below is runnable end-to-end on **synthetic sample data** included in this notebook,
so you can execute it top-to-bottom with no external documents required. Swap in your own corpus
where marked.

**Pipeline stages covered:**
1. Language-aware chunking
2. Embedding model benchmarking
3. Hybrid retrieval (sparse + dense with RRF fusion)
4. Reranking
5. Query expansion (HyDE)
6. Evaluation (MRR, Recall@k)
7. Putting it all together — a single `MultilingualRAGPipeline` class

---
**Author:** Abhishek Kumar Pandey  
**Related work:** Extractive QA over ancient scriptures (IEEE Access, 2024) · NLG survey (Neural Processing Letters, 2023)


## 0. Setup

Install dependencies. This notebook uses:
- `sentence-transformers` for embeddings and reranking
- `rank_bm25` for sparse retrieval
- `indic-nlp-library` for Indic-language-aware tokenization and morphological analysis
- `numpy` for scoring math

Run this cell once.


In [ ]:
# If running on Google Colab or a fresh environment, uncomment the line below.
# !pip install sentence-transformers rank_bm25 indic-nlp-library numpy --quiet

import logging
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Callable

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("multilingual_rag")


### Config

A small config dataclass keeps all the tunable pipeline parameters in one place — chunk size,
overlap, RRF constant, embedding/reranker model names, etc. This is the "enhanced" addition over
the inline code from the article: in a real project you'd load this from a YAML/JSON file or
environment variables rather than hardcoding it.


In [ ]:
@dataclass
class RAGConfig:
    lang_code: str = "hi"                      # ISO-ish code used by indic-nlp-library (e.g. "hi", "bn", "mr")
    max_chunk_chars: int = 800                  # target chunk size in characters
    overlap_chars: int = 100                    # overlap between consecutive chunks
    embedding_model: str = "BAAI/bge-m3"        # strong multilingual embedding baseline
    reranker_model: str = "BAAI/bge-reranker-v2-m3"
    rrf_k: int = 60                             # reciprocal rank fusion constant (standard default)
    retrieval_k: int = 10                       # candidates to retrieve per leg before fusion
    rerank_top_k: int = 5                       # final number of reranked results to return

config = RAGConfig(lang_code="hi")
logger.info(f"Loaded config: {config}")


## 1. Language-Aware Chunking

Most chunking strategies assume English-like sentence and paragraph structure. Indian languages
complicate this in two ways:

- **Script-level tokenization** behaves differently — Devanagari conjunct consonants, for example,
  can be split incorrectly by naive tokenizers, changing word meaning.
- **Morphologically rich languages** pack more grammatical information into single inflected words,
  so a chunk boundary sized "right" for English can cut a sentence at a semantically awkward point.

**Fix:** chunk on language-aware sentence boundaries (not naive punctuation splitting — the
Devanagari danda `।` is not the same as a period `.`), and keep a small character-overlap between
chunks so context isn't lost at boundaries.


In [ ]:
def simple_indic_sentence_split(text: str, lang_code: str = "hi") -> List[str]:
    """
    Lightweight sentence splitter for Indic scripts.

    In production, prefer `indic_nlp_library`'s trained sentence tokenizer:

        from indicnlp.tokenize import sentence_tokenize
        sentences = sentence_tokenize.sentence_split(text, lang=lang_code)

    This fallback handles the common Devanagari sentence terminators (।, ॥) plus
    standard punctuation, so the notebook runs without the extra dependency if needed.
    """
    import re
    # Split on Devanagari danda/double-danda, and standard sentence enders,
    # while keeping the delimiter attached to the preceding sentence.
    pattern = r'(?<=[।॥.!?])\s+'
    sentences = re.split(pattern, text.strip())
    return [s.strip() for s in sentences if s.strip()]


def chunk_indic_text(
    text: str,
    lang_code: str = "hi",
    max_chunk_chars: int = 800,
    overlap_chars: int = 100,
) -> List[str]:
    """
    Chunk text on sentence boundaries rather than a fixed character window,
    so we never cut a sentence (and therefore an inflected word) mid-way.
    Keeps a small trailing overlap between chunks for retrieval context continuity.
    """
    sentences = simple_indic_sentence_split(text, lang_code)

    chunks, current_chunk, current_len = [], [], 0

    for sent in sentences:
        sent_len = len(sent)

        if current_len + sent_len > max_chunk_chars and current_chunk:
            chunks.append(" ".join(current_chunk))

            # carry the tail of the previous chunk forward as overlap context
            overlap_sents, overlap_len = [], 0
            for s in reversed(current_chunk):
                if overlap_len + len(s) > overlap_chars:
                    break
                overlap_sents.insert(0, s)
                overlap_len += len(s)
            current_chunk, current_len = overlap_sents, overlap_len

        current_chunk.append(sent)
        current_len += sent_len

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    logger.info(f"Chunked text into {len(chunks)} chunks (lang={lang_code})")
    return chunks


### Try it — synthetic Hindi sample

Below is a small synthetic Hindi corpus so the rest of the notebook can run end-to-end without
external documents. Replace `sample_corpus` with your own documents when adapting this notebook.


In [ ]:
sample_corpus = [
    "भारत में कृत्रिम बुद्धिमत्ता का उपयोग तेजी से बढ़ रहा है। "
    "कई कंपनियां अब एजेंटिक एआई सिस्टम बना रही हैं। "
    "लेकिन अधिकांश सिस्टम अंग्रेजी भाषा के लिए ही बनाए गए हैं।",

    "रिट्रीवल-ऑगमेंटेड जनरेशन यानी RAG एक तकनीक है जो भाषा मॉडल को "
    "बाहरी दस्तावेज़ों से जानकारी खोजने में मदद करती है। "
    "यह तकनीक प्रश्न-उत्तर प्रणाली की सटीकता बढ़ाती है।",

    "कम संसाधन वाली भाषाओं जैसे भोजपुरी और मैथिली के लिए "
    "एआई मॉडल बनाना एक चुनौती है। "
    "प्रशिक्षण डेटा की कमी इसका मुख्य कारण है।",

    "DRDO एक स्वदेशी बड़ा भाषा मॉडल विकसित कर रहा है। "
    "यह मॉडल पूरी तरह से एयर-गैप्ड वातावरण में काम करेगा। "
    "इसका उपयोग साइबर सुरक्षा के लिए किया जाएगा।",
]

all_chunks = []
for doc in sample_corpus:
    all_chunks.extend(chunk_indic_text(doc, lang_code=config.lang_code,
                                        max_chunk_chars=config.max_chunk_chars,
                                        overlap_chars=config.overlap_chars))

print(f"Total chunks: {len(all_chunks)}\n")
for i, c in enumerate(all_chunks):
    print(f"[{i}] {c}\n")


## 2. Embedding Model Benchmarking

Using a general multilingual embedding model "because it supports Hindi" is not the same as it
performing *well* on Hindi. Multilingual embedding models are typically trained on wildly
imbalanced corpora — English dominates by orders of magnitude — so stated language coverage says
nothing about actual retrieval quality in a given language.

**Fix:** build a small labeled evaluation set (query → relevant document pairs) in the target
language, and benchmark candidate models against *that*, rather than trusting general leaderboards
that are mostly English/high-resource weighted.


In [ ]:
def benchmark_embedding_models(
    eval_pairs: List[Dict],
    candidate_models: List[str],
) -> Dict[str, float]:
    """
    eval_pairs: list of dicts, each: {
        "query": str,
        "relevant_doc": str,
        "irrelevant_docs": [str, ...]
    }
    candidate_models: HF model names to compare, e.g.
        ["BAAI/bge-m3", "intfloat/multilingual-e5-large"]

    Returns top-1 accuracy per model: how often the relevant doc scored highest.
    """
    from sentence_transformers import SentenceTransformer, util

    results = {}
    for model_name in candidate_models:
        logger.info(f"Benchmarking {model_name} ...")
        model = SentenceTransformer(model_name)
        correct = 0

        for pair in eval_pairs:
            q_emb = model.encode(pair["query"], convert_to_tensor=True)
            all_docs = [pair["relevant_doc"]] + pair["irrelevant_docs"]
            doc_embs = model.encode(all_docs, convert_to_tensor=True)
            scores = util.cos_sim(q_emb, doc_embs)[0]
            top_idx = int(np.argmax(scores.cpu().numpy()))
            if top_idx == 0:  # index 0 is always the relevant doc
                correct += 1

        results[model_name] = correct / len(eval_pairs)
        logger.info(f"{model_name}: top-1 accuracy = {results[model_name]:.2%}")

    return results


### Try it — tiny synthetic eval set

A real evaluation set should have 50-100+ pairs. This is a toy example (3 pairs) purely to show
the mechanics; scale it up with your own labeled data before trusting the result.


In [ ]:
tiny_eval_pairs = [
    {
        "query": "एजेंटिक एआई क्या है",
        "relevant_doc": "कई कंपनियां अब एजेंटिक एआई सिस्टम बना रही हैं।",
        "irrelevant_docs": [
            "DRDO एक स्वदेशी बड़ा भाषा मॉडल विकसित कर रहा है।",
            "प्रशिक्षण डेटा की कमी इसका मुख्य कारण है।",
        ],
    },
    {
        "query": "RAG तकनीक कैसे काम करती है",
        "relevant_doc": "यह तकनीक प्रश्न-उत्तर प्रणाली की सटीकता बढ़ाती है।",
        "irrelevant_docs": [
            "इसका उपयोग साइबर सुरक्षा के लिए किया जाएगा।",
            "भोजपुरी और मैथिली के लिए एआई मॉडल बनाना एक चुनौती है।",
        ],
    },
]

# Uncomment to actually run (downloads model weights — can be slow on first run):
# candidate_models = ["BAAI/bge-m3"]
# benchmark_results = benchmark_embedding_models(tiny_eval_pairs, candidate_models)
# print(benchmark_results)

print("Eval set ready:", len(tiny_eval_pairs), "pairs (expand to 50-100+ for real use)")


## 3. Hybrid Retrieval — Sparse (BM25) + Dense, Fused with RRF

Sparse retrieval (BM25) depends on consistent tokenization. Inflection-heavy Indian languages mean
the same root word can appear in dozens of surface forms, fragmenting term-frequency statistics
unless stemming/morphological analysis is applied.

Dense retrieval quality is bounded by the embedding model's actual competence in the target
language (Stage 2).

**Reciprocal Rank Fusion (RRF)** combines both ranked lists into one. A weak leg doesn't just
underperform — if it confidently returns *wrong* results (not just fewer results), it can drag
down the fused ranking. Test both legs separately before trusting an even-weighted fusion.


In [ ]:
def simple_indic_stem(token: str) -> str:
    """
    Lightweight fallback stemmer — strips a few common Hindi inflectional suffixes.

    In production, prefer `indic_nlp_library`'s morphological analyzer:

        from indicnlp.morph import unsupervised_morph
        analyzer = unsupervised_morph.UnsupervisedMorphAnalyzer(lang_code)
        stem = analyzer.morph_analyze(token)[0]

    This fallback is intentionally simple so the notebook runs without the extra dependency.
    """
    common_suffixes = ["ों", "ों", "ें", "ियों", "ियाँ", "ता", "ना", "ने", "की", "का", "के"]
    for suf in sorted(common_suffixes, key=len, reverse=True):
        if token.endswith(suf) and len(token) > len(suf) + 1:
            return token[: -len(suf)]
    return token


def build_indic_bm25_index(documents: List[str], lang_code: str = "hi"):
    """Build a BM25 sparse index with language-aware (stemmed) tokenization."""
    from rank_bm25 import BM25Okapi

    def stem_tokenize(text: str) -> List[str]:
        return [simple_indic_stem(tok) for tok in text.split()]

    tokenized_corpus = [stem_tokenize(doc) for doc in documents]
    bm25 = BM25Okapi(tokenized_corpus)
    logger.info(f"Built BM25 index over {len(documents)} documents")
    return bm25, stem_tokenize


def hybrid_retrieve(
    query: str,
    bm25,
    stem_tokenize: Callable,
    dense_model,
    documents: List[str],
    doc_embeddings,
    k: int = 10,
    rrf_k: int = 60,
) -> List[Tuple[str, float]]:
    """Retrieve via sparse + dense legs, fuse with reciprocal rank fusion."""
    from sentence_transformers import util

    # sparse leg
    bm25_scores = bm25.get_scores(stem_tokenize(query))
    sparse_ranking = np.argsort(bm25_scores)[::-1][:k]

    # dense leg
    q_emb = dense_model.encode(query, convert_to_tensor=True)
    dense_scores = util.cos_sim(q_emb, doc_embeddings)[0].cpu().numpy()
    dense_ranking = np.argsort(dense_scores)[::-1][:k]

    # reciprocal rank fusion: score = sum(1 / (rrf_k + rank))
    fused_scores: Dict[int, float] = {}
    for rank, idx in enumerate(sparse_ranking):
        fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (rrf_k + rank + 1)
    for rank, idx in enumerate(dense_ranking):
        fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (rrf_k + rank + 1)

    ranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return [(documents[idx], score) for idx, score in ranked[:k]]


In [ ]:
# Build the sparse index over our sample chunks
bm25_index, stem_tokenizer = build_indic_bm25_index(all_chunks, lang_code=config.lang_code)

print("BM25 index built over", len(all_chunks), "chunks.")
print("Example stemmed tokens:", stem_tokenizer(all_chunks[0])[:8])

# Dense leg requires the embedding model — uncomment to run:
# from sentence_transformers import SentenceTransformer
# dense_model = SentenceTransformer(config.embedding_model)
# doc_embeddings = dense_model.encode(all_chunks, convert_to_tensor=True)
# results = hybrid_retrieve("एजेंटिक एआई क्या है", bm25_index, stem_tokenizer,
#                            dense_model, all_chunks, doc_embeddings, k=config.retrieval_k)
# for doc, score in results:
#     print(f"{score:.4f} | {doc}")


## 4. Reranking

Cross-encoder rerankers are typically the least multilingual-aware component in a RAG stack —
reranking is more compute-intensive, so multilingual reranker options are fewer and less mature
than multilingual embedding models.

**Decision point:** use a multilingual reranker and validate it heavily on your evaluation set,
*or* skip reranking for low-resource languages and lean more heavily on a well-tuned hybrid
retrieval step. Treat this as a deliberate trade-off, not a default.


In [ ]:
def rerank(
    query: str,
    candidate_docs: List[str],
    reranker_model_name: str = "BAAI/bge-reranker-v2-m3",
) -> List[Tuple[str, float]]:
    """Re-score candidate documents against the query with a cross-encoder."""
    from sentence_transformers import CrossEncoder

    model = CrossEncoder(reranker_model_name)
    pairs = [[query, doc] for doc in candidate_docs]
    scores = model.predict(pairs)

    ranked = sorted(zip(candidate_docs, scores), key=lambda x: x[1], reverse=True)
    return ranked

# Example usage (uncomment to run — downloads reranker weights):
# candidates = [c for c, _ in results]  # from Stage 3
# reranked = rerank("एजेंटिक एआई क्या है", candidates, config.reranker_model)
# for doc, score in reranked:
#     print(f"{score:.4f} | {doc}")


## 5. Query Expansion (HyDE)

HyDE (Hypothetical Document Embeddings) generates a hypothetical answer first, then searches using
that hypothetical answer instead of the raw query — because a full answer tends to match real
documents better than a short question does.

It is only as good as the generation model's fluency in the target language. For genuinely
low-resource languages, HyDE can introduce more noise than signal if the generation model's
in-language fluency is weak.

**Recommendation:** A/B test HyDE on vs. off for your specific language — don't assume it always
helps. This is one of the more counterintuitive findings in low-resource RAG work.


In [ ]:
def hyde_expand(query: str, generation_fn: Callable, lang_code: str = "hi") -> str:
    """
    generation_fn: any callable(prompt: str) -> str, e.g. wrapping an LLM API call.
    Kept generic so you can plug in whichever generation model/API you use.
    """
    prompt = (
        f"Answer the following question in language code '{lang_code}', "
        f"as if writing a short factual passage:\n\n{query}"
    )
    return generation_fn(prompt)


def hyde_retrieve(
    query: str,
    generation_fn: Callable,
    dense_model,
    doc_embeddings,
    documents: List[str],
    lang_code: str = "hi",
    k: int = 10,
) -> List[Tuple[str, float]]:
    from sentence_transformers import util

    hyde_doc = hyde_expand(query, generation_fn, lang_code)
    hyde_emb = dense_model.encode(hyde_doc, convert_to_tensor=True)
    scores = util.cos_sim(hyde_emb, doc_embeddings)[0].cpu().numpy()
    top_idx = np.argsort(scores)[::-1][:k]
    return [(documents[i], float(scores[i])) for i in top_idx]

# Example generation_fn stub — replace with a real LLM call (Anthropic, OpenAI, local model, etc.)
def dummy_generation_fn(prompt: str) -> str:
    return "यह एक उदाहरण उत्तर है।"  # placeholder — wire up your actual LLM here

print("HyDE functions ready. Wire `dummy_generation_fn` to a real LLM call before use.")


## 6. Evaluation — the Step People Skip

None of the previous stages matter if there's no way to measure whether they're actually working.
Translating an English evaluation set is **not** equivalent to building one in-language —
translated queries often don't reflect how people actually phrase questions.

**Recommendation:** build even 50-100 manually curated query → relevant-document pairs, ideally
sourced from real user queries rather than synthetic ones. This is disproportionately valuable
compared to almost any other single investment in the pipeline.

Metrics used below:
- **MRR (Mean Reciprocal Rank):** rewards ranking the first relevant result as high as possible
- **Recall@k:** fraction of all relevant documents captured in the top-k results


In [ ]:
def evaluate_pipeline(
    eval_set: List[Dict],
    retrieve_fn: Callable,
    k: int = 5,
) -> Dict[str, float]:
    """
    eval_set: list of {"query": str, "relevant_doc_ids": [int, ...]}
    retrieve_fn: callable(query, k) -> list of (doc_text, doc_id, score)
    """
    mrr_total, recall_at_k_total = 0.0, 0.0

    for item in eval_set:
        results = retrieve_fn(item["query"], k=k)
        retrieved_ids = [r[1] for r in results]

        # Mean Reciprocal Rank — credit for the first relevant hit's rank
        for rank, doc_id in enumerate(retrieved_ids, start=1):
            if doc_id in item["relevant_doc_ids"]:
                mrr_total += 1.0 / rank
                break

        # Recall@k — fraction of relevant docs captured in top-k
        hits = len(set(retrieved_ids) & set(item["relevant_doc_ids"]))
        recall_at_k_total += hits / len(item["relevant_doc_ids"])

    n = len(eval_set)
    return {"MRR": mrr_total / n, f"Recall@{k}": recall_at_k_total / n}

print("Evaluation function ready. Wire it to your real retrieve_fn + labeled eval_set.")


## 7. Putting It All Together — `MultilingualRAGPipeline`

A single class wrapping indexing and querying, with the HyDE path optional via a flag.
This is the "production shape" — everything above as reusable building blocks assembled into one
object you'd actually import and use in an application.


In [ ]:
class MultilingualRAGPipeline:
    """
    End-to-end multilingual RAG pipeline.

    Usage:
        pipeline = MultilingualRAGPipeline(config, generation_fn=your_llm_call)
        pipeline.index(documents)
        results = pipeline.query("आपका प्रश्न यहाँ", use_hyde=False)
    """

    def __init__(self, config: RAGConfig, generation_fn: Optional[Callable] = None):
        from sentence_transformers import SentenceTransformer

        self.config = config
        self.generation_fn = generation_fn
        self.dense_model = SentenceTransformer(config.embedding_model)
        self.bm25 = None
        self.stem_tokenize = None
        self.documents: List[str] = []
        self.doc_embeddings = None
        logger.info(f"Pipeline initialized for lang='{config.lang_code}' "
                    f"with embedding model '{config.embedding_model}'")

    def index(self, raw_documents: List[str]) -> None:
        """Chunk, build sparse index, and compute dense embeddings for a document set."""
        chunked = []
        for doc in raw_documents:
            chunked.extend(chunk_indic_text(
                doc,
                lang_code=self.config.lang_code,
                max_chunk_chars=self.config.max_chunk_chars,
                overlap_chars=self.config.overlap_chars,
            ))

        self.documents = chunked
        self.bm25, self.stem_tokenize = build_indic_bm25_index(chunked, self.config.lang_code)
        self.doc_embeddings = self.dense_model.encode(chunked, convert_to_tensor=True)
        logger.info(f"Indexed {len(chunked)} chunks from {len(raw_documents)} documents")

    def query(self, query_text: str, use_hyde: bool = False) -> List[Tuple[str, float]]:
        """Retrieve (hybrid or HyDE) then rerank; returns top reranked results."""
        if use_hyde:
            if self.generation_fn is None:
                raise ValueError("HyDE requires a generation_fn to be set on the pipeline.")
            candidates = hyde_retrieve(
                query_text, self.generation_fn, self.dense_model,
                self.doc_embeddings, self.documents,
                lang_code=self.config.lang_code, k=self.config.retrieval_k,
            )
        else:
            candidates = hybrid_retrieve(
                query_text, self.bm25, self.stem_tokenize, self.dense_model,
                self.documents, self.doc_embeddings,
                k=self.config.retrieval_k, rrf_k=self.config.rrf_k,
            )

        candidate_docs = [c[0] for c in candidates]
        reranked = rerank(query_text, candidate_docs, self.config.reranker_model)
        return reranked[: self.config.rerank_top_k]


In [ ]:
# --- Demo run (uncomment to execute — downloads model weights on first run) ---

# pipeline = MultilingualRAGPipeline(config, generation_fn=dummy_generation_fn)
# pipeline.index(sample_corpus)
# top_results = pipeline.query("एजेंटिक एआई क्या है", use_hyde=False)
# for doc, score in top_results:
#     print(f"{score:.4f} | {doc}")

print("Pipeline class ready. Uncomment the demo block above to run end-to-end "
      "(requires downloading embedding + reranker model weights).")


## Closing Notes

None of this is exotic engineering — it's the same RAG architecture used everywhere else, with
deliberate attention paid at each stage instead of assuming English-tuned defaults generalize.
The gap between a multilingual RAG system that technically works and one that actually serves
Indian-language users well is almost entirely in these evaluation and tuning decisions, not in
needing fundamentally different technology.

**If you're running a RAG pipeline today — do you actually know how it performs outside English,
or are you assuming it does?**

---

### References

1. A. K. Pandey and S. S. Roy, "Extractive Question Answering Over Ancient Scriptures Texts Using
   Generative AI and Natural Language Processing Techniques," *IEEE Access*, vol. 12, 2024.
   DOI: [10.1109/ACCESS.2024.3431282](https://doi.org/10.1109/ACCESS.2024.3431282)
2. A. K. Pandey and S. S. Roy, "Natural Language Generation Using Sequential Models: A Survey,"
   *Neural Processing Letters*, vol. 55, pp. 7709–7742, 2023.
   DOI: [10.1007/s11063-023-11281-6](https://doi.org/10.1007/s11063-023-11281-6)

### Suggested repo structure

```
multilingual-rag-india/
├── README.md
├── requirements.txt
├── multilingual_rag_blueprint.ipynb   <- this notebook
└── LICENSE
```

### requirements.txt

```
sentence-transformers>=2.7.0
rank_bm25>=0.2.2
indic-nlp-library>=0.92
numpy>=1.26.0
```
